In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [5]:
from src.config import DEFAULT_STGCN_PARAMS_V1, POSE_DATASET_ROOT, DATASET_ROOT
from src.rwf2000 import RWF2000PoseDataset, RWF2000Dataset
import skelbumentations as S
import torch

train_dataset = RWF2000PoseDataset(POSE_DATASET_ROOT, split="train", max_people=3)


In [7]:
tensor, label = train_dataset[0] # [C, T, V, M]

transform = S.Compose([
    S.SelectRandomFrames(
        [S.WholeOcclusion()],
        min_num=25,
        max_num=50,
    )
])
person_index = 0

person_pose = tensor[:, :, :, person_index].clone()

# Preserve the original data for comparison
original_pose = person_pose.clone()
original_invalid = person_pose[2] == 0

# Use independent NumPy copies because the library edits in place
person_pose_np = (
    person_pose.permute(1, 2, 0)
    .cpu()
    .numpy()
    .copy()
)

invalid_np = (
    original_invalid
    .cpu()
    .numpy()
    .copy()
)

result = transform(
    keypoints=person_pose_np,
    invalid=invalid_np,
)

augmented_pose = torch.from_numpy(
    result["keypoints"]
).permute(2, 0, 1)

augmented_invalid = torch.from_numpy(result["invalid"])

new_invalid = augmented_invalid & ~original_invalid

difference = original_pose != augmented_pose
changed_frames = difference.any(dim=0).any(dim=1)

print(
    "Changed frames:",
    torch.where(changed_frames)[0].tolist(),
)

print(
    "Frames with new occlusions:",
    torch.where(new_invalid.any(dim=1))[0].tolist(),
)

print("Newly invalid joints:", new_invalid.sum().item())

Changed frames: [97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144]
Frames with new occlusions: [97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144]
Newly invalid joints: 816
